# FOXF1_bead — 09_fixed_plotting

**Feeds:** Fig 2c, ED Fig 3d, and the ED Fig 3d (right) SOX2 vs TBXT pixel density

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 09 Fixed Plotting

## Strategy

This notebook is plotting-only. It consumes the fixed quantification intermediates written by `07_fixed_quantification.ipynb` and the live FOXF1 trace intermediates from `06_live_plotting.ipynb`. The section order is organized by interpretation: first the FOXF1 decision plots, then the broader fixed bead-distance views, then the pairwise state-space plots. Within each paired view, the equal-support version comes first.

From this point onward, each numbered figure is a single image. Redundant composite figures have been removed. Figures are numbered `9.1`, `9.2`, ... in the order they appear.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

MPLCONFIGDIR = ROOT / ".matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("ROOT:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from scripts import pipeline_common as common
from scripts import live_fixed_quantification as lfq
from scipy.ndimage import gaussian_filter

## Paths And Configuration

In [ ]:
OUT_DIR = ROOT / "results/measurements/live_fixed_small"
FIGURE_DIR_ROOT = ROOT / "results/figures"
FIG_DIR = FIGURE_DIR_ROOT / "09"
CANDIDATE_FIGURE_DIR = FIG_DIR / "candidate_figures"
ALTERNATE_FIGURE_DIR = FIG_DIR / "alternate_figures"
for path in [OUT_DIR, FIG_DIR, CANDIDATE_FIGURE_DIR, ALTERNATE_FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TRACE_PLOT_YMIN = -0.5
TRACE_YLIM_SUPPORT_FRACTION = 2.0 / 3.0
TRACE_YLIM_UPPER_QUANTILE = 0.95
TRACE_YLIM_UPPER_PAD = 0.10
PAIRWISE_DENSITY_BINS = 160
PAIRWISE_DENSITY_SMOOTH_SIGMA = 1.2
PAIRWISE_DENSITY_LOWER_Q = 0.001
PAIRWISE_DENSITY_UPPER_Q = 0.9995
PAIRWISE_PANEL_CONFIG = {
    "Fixed analysis pixels | SOX2 vs FOXF1-YFP reporter": {"x_max": 0.80, "y_max": 0.35, "x_gate": 0.15, "y_gate": 0.12},
    "Fixed analysis pixels | SOX2 vs TBXT": {"x_max": 0.65, "y_max": 0.60, "x_gate": 0.18, "y_gate": 0.23},
}

MEASUREMENT_ORDER = [
    "fixed_tagyfp_bgz_over_dapi_gate",
    "fixed_sox2_bgz_over_dapi_gate",
    "fixed_t_bgz_over_dapi_gate",
]
MEASUREMENT_LABELS = {
    "fixed_tagyfp_bgz_over_dapi_gate": "FOXF1-YFP reporter (fixed) / DAPI",
    "fixed_sox2_bgz_over_dapi_gate": "SOX2 / DAPI",
    "fixed_t_bgz_over_dapi_gate": "TBXT / DAPI",
}
MEASUREMENT_COLORS = {
    "fixed_tagyfp_bgz_over_dapi_gate": "#c62828",
    "fixed_sox2_bgz_over_dapi_gate": "#b58900",
    "fixed_t_bgz_over_dapi_gate": "#2b6cb0",
}

FIXED_RATIO_PIXEL_BIN_STATS_TSV = OUT_DIR / "fixed_ratio_pixel_bin_stats.tsv"
FIXED_RATIO_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_ratio_distance_trace_across_images.tsv"
FIXED_RATIO_TRACE_ALL_PIXELS_TSV = OUT_DIR / "fixed_ratio_distance_trace_all_pixels_merged.tsv"
FIXED_RATIO_TRACE_EQUAL_SUPPORT_TSV = OUT_DIR / "fixed_ratio_distance_trace_equal_support.tsv"
FIXED_RATIO_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_ratio_distance_trace_equal_support_across_images.tsv"
FIXED_MEASUREMENT_SUMMARY_TXT = OUT_DIR / "fixed_measurement_summary.txt"
FIXED_RATIO_ANALYSIS_PIXELS_NPZ = OUT_DIR / "fixed_ratio_analysis_pixels.npz"

FIXED_FOXF1_PIXEL_BIN_STATS_TSV = OUT_DIR / "fixed_foxf1_pixel_bin_stats.tsv"
FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_foxf1_distance_trace_across_images.tsv"
FIXED_FOXF1_TRACE_ALL_PIXELS_TSV = OUT_DIR / "fixed_foxf1_distance_trace_all_pixels_merged.tsv"
FIXED_FOXF1_TRACE_EQUAL_SUPPORT_TSV = OUT_DIR / "fixed_foxf1_distance_trace_equal_support.tsv"
FIXED_FOXF1_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "fixed_foxf1_distance_trace_equal_support_across_images.tsv"

LIVE_TRACE_ACROSS_IMAGES_TSV = OUT_DIR / "live_tagyfp_distance_trace_across_images.tsv"
LIVE_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV = OUT_DIR / "live_tagyfp_distance_trace_equal_support_across_images.tsv"

FIXED_RATIO_TRACE_COMPARISON_PNG = ALTERNATE_FIGURE_DIR / "fixed_ratio_distance_trace_comparison.png"
FIXED_RATIO_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_ratio_distance_plot_fixed_width.png"
FIXED_RATIO_TRACE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "fixed_ratio_distance_plot_equal_support.png"
FIXED_ANALYSIS_SUPPORT_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_analysis_support_fixed_width.png"
FIXED_FOXF1_MERGED_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_all_pixels_merged_fixed_width.png"
FIXED_FOXF1_MERGED_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_all_pixels_merged_equal_support.png"
FIXED_RATIO_MERGED_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_ratio_all_pixels_merged_fixed_width.png"
FIXED_RATIO_MERGED_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "fixed_ratio_all_pixels_merged_equal_support.png"
FIXED_FOXF1_TRACE_COMPARISON_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_distance_trace_comparison.png"
FIXED_FOXF1_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_distance_plot_fixed_width.png"
FIXED_FOXF1_TRACE_FIXED_WIDTH_HIGHLIGHTED_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_distance_plot_fixed_width_highlighted_outliers.png"
FIXED_FOXF1_TRACE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "fixed_foxf1_distance_plot_equal_support.png"
FOXF1_LIVE_VS_FIXED_COMPARISON_PNG = ALTERNATE_FIGURE_DIR / "foxf1_live_vs_fixed_comparison.png"
FOXF1_LIVE_VS_FIXED_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "foxf1_live_vs_fixed_fixed_width.png"
FOXF1_LIVE_VS_FIXED_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "foxf1_live_vs_fixed_equal_support.png"
FIXED_SINGLE_CHANNEL_TRACE_GRID_PNG = ALTERNATE_FIGURE_DIR / "fixed_single_channel_distance_plots.png"
FIXED_TAGYFP_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_single_channel_fixed_width.png"
FIXED_TAGYFP_TRACE_EQUAL_SUPPORT_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_single_channel_equal_support.png"
FIXED_SOX2_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_sox2_single_channel_fixed_width.png"
FIXED_SOX2_TRACE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "fixed_sox2_single_channel_equal_support.png"
FIXED_T_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_t_single_channel_fixed_width.png"
FIXED_T_TRACE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "fixed_t_single_channel_equal_support.png"
FIXED_FOXF1_SOX2_TRACE_FIXED_WIDTH_PNG = ALTERNATE_FIGURE_DIR / "fixed_foxf1_sox2_distance_plot_fixed_width.png"
FIXED_FOXF1_SOX2_TRACE_EQUAL_SUPPORT_PNG = CANDIDATE_FIGURE_DIR / "fixed_foxf1_sox2_distance_plot_equal_support.png"
FIXED_PAIRWISE_DENSITY_PNG = ALTERNATE_FIGURE_DIR / "fixed_pairwise_density_plots.png"
FIXED_FOXF1_SOX2_DENSITY_PNG = CANDIDATE_FIGURE_DIR / "fixed_foxf1_sox2_density.png"
FIXED_SOX2_T_DENSITY_PNG = CANDIDATE_FIGURE_DIR / "fixed_sox2_t_density.png"


def _save_plot_all_formats(fig, png_path: Path, dpi: int = 180, bbox_inches: str = "tight") -> None:
    png_path = Path(png_path)
    fig.savefig(png_path, dpi=dpi, bbox_inches=bbox_inches)
    fig.savefig(png_path.with_suffix(".pdf"), bbox_inches=bbox_inches)
    fig.savefig(png_path.with_suffix(".svg"), bbox_inches=bbox_inches)


## Load Shared Plotting Intermediates

In [ ]:
required_outputs = [
    FIXED_RATIO_PIXEL_BIN_STATS_TSV,
    FIXED_RATIO_TRACE_ACROSS_IMAGES_TSV,
    FIXED_RATIO_TRACE_ALL_PIXELS_TSV,
    FIXED_RATIO_TRACE_EQUAL_SUPPORT_TSV,
    FIXED_RATIO_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV,
    FIXED_MEASUREMENT_SUMMARY_TXT,
    FIXED_RATIO_ANALYSIS_PIXELS_NPZ,
    FIXED_FOXF1_PIXEL_BIN_STATS_TSV,
    FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV,
    FIXED_FOXF1_TRACE_ALL_PIXELS_TSV,
    FIXED_FOXF1_TRACE_EQUAL_SUPPORT_TSV,
    FIXED_FOXF1_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV,
    LIVE_TRACE_ACROSS_IMAGES_TSV,
    LIVE_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV,
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError("Run 06 and 07 first. Missing outputs:\n" + "\n".join(missing))

fixed_ratio_stats_df = pd.read_csv(FIXED_RATIO_PIXEL_BIN_STATS_TSV, sep="	")
fixed_ratio_trace_df = pd.read_csv(FIXED_RATIO_TRACE_ACROSS_IMAGES_TSV, sep="	")
fixed_ratio_merged_df = pd.read_csv(FIXED_RATIO_TRACE_ALL_PIXELS_TSV, sep="	")
fixed_ratio_equal_support_df = pd.read_csv(FIXED_RATIO_TRACE_EQUAL_SUPPORT_TSV, sep="	")
fixed_ratio_equal_support_across_images_df = pd.read_csv(FIXED_RATIO_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="	")

foxf1_stats_df = pd.read_csv(FIXED_FOXF1_PIXEL_BIN_STATS_TSV, sep="	")
foxf1_trace_df = pd.read_csv(FIXED_FOXF1_TRACE_ACROSS_IMAGES_TSV, sep="	")
foxf1_merged_df = pd.read_csv(FIXED_FOXF1_TRACE_ALL_PIXELS_TSV, sep="	")
foxf1_equal_support_df = pd.read_csv(FIXED_FOXF1_TRACE_EQUAL_SUPPORT_TSV, sep="	")
foxf1_equal_support_across_images_df = pd.read_csv(FIXED_FOXF1_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="	")

live_fixedwidth_df = pd.read_csv(LIVE_TRACE_ACROSS_IMAGES_TSV, sep="	")
live_equal_support_df = pd.read_csv(LIVE_TRACE_EQUAL_SUPPORT_ACROSS_IMAGES_TSV, sep="	")

with np.load(FIXED_RATIO_ANALYSIS_PIXELS_NPZ) as npz:
    foxf1_vals = np.asarray(npz["foxf1_ratio"], dtype=np.float32)
    sox2_vals_for_foxf1_pair = np.asarray(npz["sox2_ratio_for_foxf1_pair"], dtype=np.float32) if "sox2_ratio_for_foxf1_pair" in npz.files else np.asarray(npz["sox2_ratio"], dtype=np.float32)
    sox2_vals = np.asarray(npz["sox2_ratio"], dtype=np.float32)
    sox2_vals_for_t_pair = np.asarray(npz["sox2_ratio_for_t_pair"], dtype=np.float32) if "sox2_ratio_for_t_pair" in npz.files else np.asarray(npz["sox2_ratio"], dtype=np.float32)
    t_vals = np.asarray(npz["t_ratio"], dtype=np.float32)
    t_vals_for_sox2_pair = np.asarray(npz["t_ratio_for_sox2_pair"], dtype=np.float32) if "t_ratio_for_sox2_pair" in npz.files else np.asarray(npz["t_ratio"], dtype=np.float32)

summary_map = {}
for line in FIXED_MEASUREMENT_SUMMARY_TXT.read_text().splitlines():
    if not line.strip() or "	" not in line:
        continue
    key, value = line.split("	", 1)
    summary_map[key] = value

equal_support_target_px = int(float(summary_map.get("equal_support_target_pixels", "1")))


MULTICHANNEL_TRACE_XMIN = 50.0
MULTICHANNEL_TRACE_XMAX = 700.0

def _normalized_trace_inputs(
    per_image_df: pd.DataFrame,
    agg_df: pd.DataFrame,
    mean_col: str,
    sd_col: str,
    x_col: str = "bin_mid_um",
    peak_xmax: float | None = None,
    peak_xmin: float | None = None,
):
    per_norm = per_image_df.copy()
    agg_norm = agg_df.copy()
    norm_rows = []
    for meas_name in MEASUREMENT_ORDER:
        sub = agg_norm[agg_norm["measurement_name"] == meas_name].copy()
        if peak_xmin is not None and x_col in sub.columns:
            sub = sub[sub[x_col].astype(float) >= float(peak_xmin)].copy()
        if peak_xmax is not None and x_col in sub.columns:
            sub = sub[sub[x_col].astype(float) <= float(peak_xmax)].copy()
        vals = sub[mean_col].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        peak = float(np.nanmax(vals)) if vals.size else 1.0
        if not np.isfinite(peak) or peak <= 1e-8:
            peak = 1.0
        norm_rows.append({"measurement_name": meas_name, "plot_peak_scale": peak})
        per_mask = per_norm["measurement_name"] == meas_name
        agg_mask = agg_norm["measurement_name"] == meas_name
        for col in ["mean_value", "median_value", "std_value", "sem_value"]:
            if col in per_norm.columns:
                per_norm.loc[per_mask, col] = per_norm.loc[per_mask, col].astype(float) / peak
        for col in [mean_col, sd_col, "sem"]:
            if col in agg_norm.columns:
                agg_norm.loc[agg_mask, col] = agg_norm.loc[agg_mask, col].astype(float) / peak
    return per_norm, agg_norm, pd.DataFrame(norm_rows)

main_agg = fixed_ratio_trace_df.copy().sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)
alt_agg = fixed_ratio_equal_support_across_images_df.copy().sort_values(["measurement_name", "bin_mid_um"]).reset_index(drop=True)
plot_fixedwidth_per_df, plot_main_agg, fixedwidth_norm_df = _normalized_trace_inputs(
    per_image_df=fixed_ratio_stats_df,
    agg_df=main_agg,
    mean_col="mean",
    sd_col="sd",
)
_, plot_alt_agg, equal_support_norm_df = _normalized_trace_inputs(
    per_image_df=fixed_ratio_stats_df,
    agg_df=alt_agg,
    mean_col="mean",
    sd_col="sd",
)
plot_fixedwidth_per_df_window, plot_main_agg_window, fixedwidth_window_norm_df = _normalized_trace_inputs(
    per_image_df=fixed_ratio_stats_df,
    agg_df=main_agg,
    mean_col="mean",
    sd_col="sd",
    peak_xmin=MULTICHANNEL_TRACE_XMIN,
    peak_xmax=MULTICHANNEL_TRACE_XMAX,
)
_, plot_alt_agg_window, equal_support_window_norm_df = _normalized_trace_inputs(
    per_image_df=fixed_ratio_stats_df,
    agg_df=alt_agg,
    mean_col="mean",
    sd_col="sd",
    peak_xmin=MULTICHANNEL_TRACE_XMIN,
    peak_xmax=MULTICHANNEL_TRACE_XMAX,
)
plot_fixedwidth_merged = fixed_ratio_merged_df.copy()
for meas_name in MEASUREMENT_ORDER:
    peak = float(np.nanmax(plot_fixedwidth_merged.loc[plot_fixedwidth_merged["measurement_name"] == meas_name, "weighted_mean_value"].to_numpy(dtype=float)))
    if not np.isfinite(peak) or peak <= 1e-8:
        peak = 1.0
    mask = plot_fixedwidth_merged["measurement_name"] == meas_name
    for col in ["weighted_mean_value", "pooled_sd_value", "pooled_sem_value"]:
        plot_fixedwidth_merged.loc[mask, col] = plot_fixedwidth_merged.loc[mask, col].astype(float) / peak
plot_equal_support_merged = fixed_ratio_equal_support_df.copy()
for meas_name in MEASUREMENT_ORDER:
    peak = float(np.nanmax(plot_equal_support_merged.loc[plot_equal_support_merged["measurement_name"] == meas_name, "mean_value"].to_numpy(dtype=float)))
    if not np.isfinite(peak) or peak <= 1e-8:
        peak = 1.0
    mask = plot_equal_support_merged["measurement_name"] == meas_name
    for col in ["mean_value", "median_value", "std_value", "sem_value"]:
        plot_equal_support_merged.loc[mask, col] = plot_equal_support_merged.loc[mask, col].astype(float) / peak

max_n_images = int(max(plot_main_agg_window["n_images"].max() if len(plot_main_agg_window) else 0, plot_alt_agg_window["n_images"].max() if len(plot_alt_agg_window) else 0))
support_min_images = max(1, int(np.ceil(TRACE_YLIM_SUPPORT_FRACTION * max_n_images)))
upper_candidates = []
for agg_df, mean_col, sd_col in [
    (plot_main_agg_window, "mean", "sd"),
    (plot_alt_agg_window, "mean", "sd"),
]:
    support_sub = agg_df.copy()
    if "bin_mid_um" in support_sub.columns:
        support_sub = support_sub[
            support_sub["bin_mid_um"].astype(float).between(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX, inclusive="both")
        ].copy()
    if len(support_sub) == 0:
        continue
    support_sub = support_sub[support_sub["n_images"] >= support_min_images].copy()
    if len(support_sub) == 0:
        continue
    upper_candidates.extend((support_sub[mean_col].astype(float) + 1.96 * support_sub["sem"].astype(float)).tolist())
if upper_candidates:
    upper_ref = float(np.nanquantile(np.asarray(upper_candidates, dtype=float), TRACE_YLIM_UPPER_QUANTILE))
    shared_ymax = float(np.ceil((upper_ref + TRACE_YLIM_UPPER_PAD) * 10.0) / 10.0)
    shared_ymax = max(shared_ymax, 1.0)
    shared_ylim = (float(TRACE_PLOT_YMIN), shared_ymax)
else:
    shared_ylim = (float(TRACE_PLOT_YMIN), 1.2)

live_fixedwidth_plot_df = live_fixedwidth_df.sort_values("bin_mid_um").reset_index(drop=True)
live_equal_support_plot_df = live_equal_support_df.sort_values("bin_mid_um").reset_index(drop=True)
fixed_fixedwidth_plot_df = foxf1_trace_df.sort_values("bin_mid_um").reset_index(drop=True)
fixed_equal_support_plot_df = foxf1_equal_support_across_images_df.sort_values("bin_mid_um").reset_index(drop=True)

single_channel_ylim_window_by_meas = {
    "fixed_sox2_bgz_over_dapi_gate": (0.0, 1.1),
    "fixed_t_bgz_over_dapi_gate": (0.5, 1.1),
}

foxf1_ymins = []
foxf1_ymaxs = []
for agg_df, mean_col, sd_col in [
    (foxf1_trace_df, "mean", "sd"),
    (foxf1_equal_support_across_images_df, "mean", "sd"),
]:
    if len(agg_df) > 0:
        foxf1_ymins.append(float(np.nanmin(agg_df[mean_col] - agg_df[sd_col])))
        foxf1_ymaxs.append(float(np.nanmax(agg_df[mean_col] + agg_df[sd_col])))
if foxf1_ymins and foxf1_ymaxs:
    y_lo = float(np.nanmin(foxf1_ymins))
    y_hi = float(np.nanmax(foxf1_ymaxs))
    y_span = max(1e-6, y_hi - y_lo)
    foxf1_shared_ylim = (y_lo - 0.06 * y_span, y_hi + 0.02 * y_span)
else:
    foxf1_shared_ylim = None

print("Loaded fixed plotting intermediates.")
print("Equal-support target pixels/bin:", f"{equal_support_target_px:,}")
print("Fixed ratio rows:", len(fixed_ratio_stats_df))
print("Pooled pairwise pixels:", f"FOXF1={len(foxf1_vals):,}, FOXF1-paired SOX2={len(sox2_vals_for_foxf1_pair):,}, SOX2={len(sox2_vals):,}, TBXT={len(t_vals):,}")


## Helper Functions

In [ ]:
def _plot_multi_channel_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, title: str, ylabel: str) -> None:
    for meas_name in MEASUREMENT_ORDER:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        per_sub = per_image_df[per_image_df["measurement_name"] == meas_name].copy()
        agg_sub = agg_df[agg_df["measurement_name"] == meas_name].copy()
        for _, sub in per_sub.groupby("canonical_position", sort=True):
            sub = sub.sort_values("bin_mid_um")
            ax.plot(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.10, linewidth=0.8)
            ax.scatter(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.06, s=6)
        if len(agg_sub) > 0:
            agg_sub = agg_sub.sort_values("bin_mid_um")
            mean_vals = agg_sub["mean"].to_numpy(dtype=float)
            ci_vals = 1.96 * agg_sub["sem"].to_numpy(dtype=float)
            ax.plot(agg_sub["bin_mid_um"], mean_vals, color=color, linewidth=2.4, label=label)
            ax.fill_between(agg_sub["bin_mid_um"], mean_vals - ci_vals, mean_vals + ci_vals, color=color, alpha=0.18)
    ax.set_xlabel("Distance from nearest bead (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best")


def _plot_multi_channel_merged(
    ax,
    merged_df: pd.DataFrame,
    title: str,
    ylabel: str,
    mean_col: str,
    sd_col: str,
    count_col: str | None = None,
    band_mode: str = "ci95",
) -> None:
    for meas_name in MEASUREMENT_ORDER:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        sub = merged_df[merged_df["measurement_name"] == meas_name].copy().sort_values("bin_mid_um")
        if len(sub) == 0:
            continue
        mean_vals = sub[mean_col].to_numpy(dtype=float)
        if band_mode == "sd":
            band_vals = sub[sd_col].to_numpy(dtype=float)
        elif band_mode == "ci95":
            if "pooled_sem_value" in sub.columns:
                band_vals = 1.96 * sub["pooled_sem_value"].to_numpy(dtype=float)
            elif "sem_value" in sub.columns:
                band_vals = 1.96 * sub["sem_value"].to_numpy(dtype=float)
            else:
                band_vals = sub[sd_col].to_numpy(dtype=float)
        else:
            raise ValueError(f"Unsupported band_mode: {band_mode}")
        ax.plot(sub["bin_mid_um"], mean_vals, color=color, linewidth=2.3, marker="o", markersize=2.6, label=label)
        ax.fill_between(sub["bin_mid_um"], mean_vals - band_vals, mean_vals + band_vals, color=color, alpha=0.15)
    ax.set_xlabel("Distance from nearest bead (um)" if "equal-support" not in title.lower() else "Mean distance within equal-support bin (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="upper left")


def _plot_fixed_foxf1_trace(
    ax,
    per_image_df,
    agg_df,
    x_col,
    y_col,
    band_col,
    title,
    color=MEASUREMENT_COLORS["fixed_tagyfp_bgz_over_dapi_gate"],
    x_label="Distance from nearest bead (um)",
    ylabel="Normalized mean standardized FOXF1-YFP reporter (fixed) / raw DAPI\n(channel peak bin = 1)",
    aggregate_label="Pooled-pixel mean",
    band_label="Pooled mean 95% CI",
    per_image_label="Per-image traces",
):
    first_gray = True
    for _, sub in per_image_df.groupby("canonical_position", sort=True):
        sub = sub.sort_values("bin_mid_um")
        ax.plot(
            sub["bin_mid_um"],
            sub["mean_value"],
            color="0.65",
            alpha=0.14,
            linewidth=0.8,
            label=per_image_label if first_gray else None,
        )
        ax.scatter(sub["bin_mid_um"], sub["mean_value"], color="0.65", alpha=0.09, s=6)
        first_gray = False
    if len(agg_df) > 0:
        agg_df = agg_df.sort_values(x_col)
        mean_vals = agg_df[y_col].to_numpy(dtype=float)
        band_vals = agg_df[band_col].to_numpy(dtype=float)
        ax.plot(agg_df[x_col], mean_vals, color=color, linewidth=2.6, label=aggregate_label)
        ax.fill_between(agg_df[x_col], mean_vals - band_vals, mean_vals + band_vals, color=color, alpha=0.22, label=band_label)
    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best", frameon=False)


def _support_aware_overlay_ylim(agg_df: pd.DataFrame, mean_col: str = "mean", sd_col: str = "sd", n_images_col: str = "n_images"):
    if len(agg_df) == 0:
        return None, 0
    max_n = int(np.nanmax(agg_df[n_images_col].to_numpy(dtype=float))) if n_images_col in agg_df.columns else 0
    support_min = max(1, int(np.ceil(TRACE_YLIM_SUPPORT_FRACTION * max_n))) if max_n > 0 else 1
    support_sub = agg_df[agg_df[n_images_col] >= support_min].copy() if n_images_col in agg_df.columns else agg_df.copy()
    if len(support_sub) == 0:
        support_sub = agg_df.copy()
    upper_candidates = (support_sub[mean_col].astype(float) + support_sub[sd_col].astype(float)).to_numpy(dtype=float)
    upper_candidates = upper_candidates[np.isfinite(upper_candidates)]
    if upper_candidates.size == 0:
        upper = 1.0
    else:
        upper = float(np.nanquantile(upper_candidates, TRACE_YLIM_UPPER_QUANTILE))
        upper = upper + TRACE_YLIM_UPPER_PAD * max(abs(upper), 1e-6)
    upper = max(upper, 1e-6)
    return (0.0, upper), support_min


def _plot_live_vs_fixed_overlay(ax, live_df: pd.DataFrame, fixed_df: pd.DataFrame, title: str, x_label: str):
    live_ylim, live_support_min = _support_aware_overlay_ylim(live_df)
    fixed_ylim, fixed_support_min = _support_aware_overlay_ylim(fixed_df)
    ax2 = ax.twinx()
    live_mean = live_df["mean"].to_numpy(dtype=float)
    live_ci = 1.96 * live_df["sem"].to_numpy(dtype=float)
    fixed_mean = fixed_df["mean"].to_numpy(dtype=float)
    fixed_ci = 1.96 * fixed_df["sem"].to_numpy(dtype=float)
    line_live, = ax.plot(live_df["bin_mid_um"], live_mean, color="#ef5350", lw=2.4, label="FOXF1-YFP reporter activity (live)", zorder=3)
    ax.fill_between(live_df["bin_mid_um"], live_mean - live_ci, live_mean + live_ci, color="#ef5350", alpha=0.18, zorder=2, label="Live 95% CI")
    line_fixed, = ax2.plot(fixed_df["bin_mid_um"], fixed_mean, color="#8e0000", lw=2.4, label="FOXF1-YFP reporter (fixed) / DAPI", zorder=3)
    ax2.fill_between(fixed_df["bin_mid_um"], fixed_mean - fixed_ci, fixed_mean + fixed_ci, color="#8e0000", alpha=0.16, zorder=2, label="Fixed 95% CI")
    if live_ylim is not None:
        ax.set_ylim(*live_ylim)
    if fixed_ylim is not None:
        ax2.set_ylim(*fixed_ylim)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Live FOXF1-YFP reporter activity (flat-field corrected, per-image bg-subtracted)", color="#ef5350")
    ax2.set_ylabel("FOXF1-YFP reporter (fixed) / raw DAPI", color="#8e0000")
    ax.tick_params(axis="y", colors="#ef5350")
    ax2.tick_params(axis="y", colors="#8e0000")
    handles = [line_live, line_fixed]
    ax.legend(handles, [h.get_label() for h in handles], loc="upper right", frameon=False)
    ax.text(0.02, 0.95, f"Live axis ~0-1 scaled (support cutoff {live_support_min} images)\nFixed axis ~0-1 scaled (support cutoff {fixed_support_min} images)", transform=ax.transAxes, ha="left", va="top", fontsize=8, bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.92})
    return ax, ax2


def _plot_single_fixed_channel_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, meas_name: str, title: str, ylabel: str, color: str):
    per_sub = per_image_df[per_image_df["measurement_name"] == meas_name].copy().sort_values(["canonical_position", "bin_mid_um"])
    agg_sub = agg_df[agg_df["measurement_name"] == meas_name].copy().sort_values("bin_mid_um")
    first_gray = True
    for _, sub in per_sub.groupby("canonical_position", sort=True):
        sub = sub.sort_values("bin_mid_um")
        ax.plot(sub["bin_mid_um"], sub["mean_value"], color="0.75", lw=0.8, alpha=0.45, zorder=1, label="Per-image traces" if first_gray else None)
        ax.scatter(sub["bin_mid_um"], sub["mean_value"], color="0.75", s=6, alpha=0.09, zorder=1)
        first_gray = False
    if len(agg_sub) > 0:
        mean_vals = agg_sub["mean"].to_numpy(dtype=float)
        ci_vals = 1.96 * agg_sub["sem"].to_numpy(dtype=float)
        ax.plot(agg_sub["bin_mid_um"], mean_vals, color=color, lw=2.4, zorder=3, label=MEASUREMENT_LABELS[meas_name])
        ax.fill_between(agg_sub["bin_mid_um"], mean_vals - ci_vals, mean_vals + ci_vals, color=color, alpha=0.18, zorder=2, label="Mean ± 95% CI")
    ax.set_xlabel("Distance from nearest bead (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="best")


def _plot_subset_channel_trace(ax, per_image_df: pd.DataFrame, agg_df: pd.DataFrame, measurement_order: list[str], title: str, ylabel: str) -> None:
    for meas_name in measurement_order:
        color = MEASUREMENT_COLORS[meas_name]
        label = MEASUREMENT_LABELS[meas_name]
        per_sub = per_image_df[per_image_df["measurement_name"] == meas_name].copy()
        agg_sub = agg_df[agg_df["measurement_name"] == meas_name].copy()
        for _, sub in per_sub.groupby("canonical_position", sort=True):
            sub = sub.sort_values("bin_mid_um")
            ax.plot(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.10, linewidth=0.8)
            ax.scatter(sub["bin_mid_um"], sub["mean_value"], color=color, alpha=0.06, s=6)
        if len(agg_sub) > 0:
            agg_sub = agg_sub.sort_values("bin_mid_um")
            mean_vals = agg_sub["mean"].to_numpy(dtype=float)
            ci_vals = 1.96 * agg_sub["sem"].to_numpy(dtype=float)
            ax.plot(agg_sub["bin_mid_um"], mean_vals, color=color, linewidth=2.4, label=label)
            ax.fill_between(agg_sub["bin_mid_um"], mean_vals - ci_vals, mean_vals + ci_vals, color=color, alpha=0.18)
    ax.set_xlabel("Distance from nearest bead (um)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best", frameon=False)


def _prepare_pairwise_density_inputs(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x_raw = np.asarray(x, dtype=float)
    y_raw = np.asarray(y, dtype=float)
    keep = np.isfinite(x_raw) & np.isfinite(y_raw) & (x_raw > 0) & (y_raw > 0)
    return x_raw[keep], y_raw[keep]


def _pairwise_base_label(axis_label: str) -> str:
    label = str(axis_label)
    if "FOXF1" in label:
        return "FOXF1"
    if "SOX2" in label:
        return "SOX2"
    if "TBXT" in label:
        return "TBXT"
    return label.split("/")[0].strip()


def _pairwise_normalize_to_unit(arr: np.ndarray) -> tuple[np.ndarray, float]:
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return arr, 1.0
    upper = float(np.nanquantile(arr, PAIRWISE_DENSITY_UPPER_Q))
    if not np.isfinite(upper) or upper <= 0:
        upper = float(np.nanmax(arr)) if arr.size else 1.0
    upper = max(upper, 1e-8)
    return np.clip(arr / upper, 0.0, 1.0), upper


def _pairwise_label_box(ax, x0: float, y0: float, text: str, ha: str = 'center', va: str = 'center') -> None:
    ax.text(
        x0,
        y0,
        text,
        ha=ha,
        va=va,
        fontsize=8,
        color='0.15',
        bbox={'facecolor': (1, 1, 1, 0.50), 'edgecolor': (0, 0, 0, 0.18), 'boxstyle': 'round,pad=0.20'},
        zorder=12,
    )


def _plot_pairwise_density(ax, x: np.ndarray, y: np.ndarray, xlabel: str, ylabel: str, title: str):
    cfg = PAIRWISE_PANEL_CONFIG.get(str(title), {})
    x_max = float(cfg.get('x_max', 1.0))
    y_max = float(cfg.get('y_max', 1.0))
    x_gate = float(cfg.get('x_gate', 0.30))
    y_gate = float(cfg.get('y_gate', 0.30))
    x_pos, y_pos = _prepare_pairwise_density_inputs(x, y)
    x, x_scale = _pairwise_normalize_to_unit(x_pos)
    y, y_scale = _pairwise_normalize_to_unit(y_pos)
    in_view = (x <= x_max) & (y <= y_max)
    x_view = x[in_view]
    y_view = y[in_view]
    hist, xedges, yedges = np.histogram2d(x_view, y_view, bins=int(PAIRWISE_DENSITY_BINS), range=[[0.0, x_max], [0.0, y_max]])
    if np.nanmax(hist) > 0:
        hist_norm = hist / float(np.nanmax(hist))
    else:
        hist_norm = hist
    im = ax.imshow(hist_norm.T, origin='lower', extent=[0.0, x_max, 0.0, y_max], aspect='auto', cmap='magma', interpolation='nearest', vmin=0.0, vmax=1.0)
    smooth = gaussian_filter(hist_norm.astype(np.float32), sigma=float(PAIRWISE_DENSITY_SMOOTH_SIGMA))
    positive = smooth[smooth > 0]
    if positive.size:
        levels = np.quantile(positive, [0.60, 0.75, 0.88, 0.95, 0.985])
        levels = np.unique(levels[np.isfinite(levels)])
        if levels.size:
            xcent = 0.5 * (xedges[:-1] + xedges[1:])
            ycent = 0.5 * (yedges[:-1] + yedges[1:])
            ax.contour(xcent, ycent, smooth.T, levels=levels, colors='0.35', linewidths=0.7, alpha=0.95)
    ax.axvline(x_gate, color='0.45', linestyle='--', linewidth=0.9, alpha=0.9)
    ax.axhline(y_gate, color='0.45', linestyle='--', linewidth=0.9, alpha=0.9)
    x_high = x_view > x_gate
    y_high = y_view > y_gate
    quadrant_fracs = {
        'hh': 100.0 * np.mean(x_high & y_high) if len(x_view) else 0.0,
        'hl': 100.0 * np.mean(x_high & ~y_high) if len(x_view) else 0.0,
        'lh': 100.0 * np.mean(~x_high & y_high) if len(x_view) else 0.0,
        'll': 100.0 * np.mean(~x_high & ~y_high) if len(x_view) else 0.0,
    }
    x_base = _pairwise_base_label(xlabel)
    y_base = _pairwise_base_label(ylabel)
    x_left = max(0.18 * x_max, 0.5 * x_gate)
    x_right = min(0.82 * x_max, x_gate + 0.5 * (x_max - x_gate))
    y_low = max(0.12 * y_max, 0.5 * y_gate)
    y_high_pos = min(0.88 * y_max, y_gate + 0.5 * (y_max - y_gate))
    _pairwise_label_box(ax, x_right, y_high_pos, f"{x_base}-high / {y_base}-high\n{quadrant_fracs['hh']:.1f}%")
    _pairwise_label_box(ax, x_right, y_low, f"{x_base}-high / {y_base}-low\n{quadrant_fracs['hl']:.1f}%")
    _pairwise_label_box(ax, x_left, y_high_pos, f"{x_base}-low / {y_base}-high\n{quadrant_fracs['lh']:.1f}%")
    _pairwise_label_box(ax, x_left, y_low, f"{x_base}-low / {y_base}-low\n{quadrant_fracs['ll']:.1f}%")
    ax.set_xlim(0.0, x_max)
    ax.set_ylim(0.0, y_max)
    ax.set_xlabel(f"{xlabel} (robustly normalized)")
    ax.set_ylabel(f"{ylabel} (robustly normalized)")
    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Normalized pixel density')
    ax.text(0.02, 0.96, f"Pixels = {len(x_view):,}\nBins = {int(PAIRWISE_DENSITY_BINS)} x {int(PAIRWISE_DENSITY_BINS)}\nboth channels > 0 only\naxis upper scale = {PAIRWISE_DENSITY_UPPER_Q:.4f} quantile\nbiology gates: x={x_gate:.2f}, y={y_gate:.2f}", transform=ax.transAxes, ha='left', va='top', fontsize=8, bbox={'facecolor': 'white', 'edgecolor': '0.85', 'boxstyle': 'round,pad=0.25', 'alpha': 0.92}, zorder=11)


def _label_panel(ax, panel_id: str) -> None:
    ax.text(
        -0.02,
        1.02,
        str(panel_id),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold",
        color="black",
        bbox={"facecolor": "white", "edgecolor": "0.2", "boxstyle": "round,pad=0.20", "alpha": 0.95},
        zorder=20,
    )


## FOXF1 Decision Support

These are the plots most directly useful for deciding whether we want to emphasize live FOXF1 or fixed FOXF1 in the manuscript-style figure set.

### Fixed FOXF1 Only

Fixed FOXF1 plotted in the same general style as the live notebook, with the main equal-support view first, the fixed-width companion, a fixed-width trace-review copy, and the two all-pixels-merged summaries with equal-support prioritized.

In [ ]:
FOXF1_FIXED_MEAS_NAME = "fixed_tagyfp_bgz_over_dapi_gate"
FOXF1_FIXED_LABEL = "FOXF1-YFP reporter (fixed) / DAPI"
FOXF1_XMAX = 700.0
FOXF1_DIP_WINDOW_UM = (700.0, 800.0)
FOXF1_REVIEW_POSITION = "5-5"
FOXF1_REVIEW_XMAX = FOXF1_XMAX

foxf1_fixedwidth_peak = float(np.nanmax(foxf1_trace_df["mean"].to_numpy(dtype=float))) if len(foxf1_trace_df) else 1.0
if not np.isfinite(foxf1_fixedwidth_peak) or foxf1_fixedwidth_peak <= 1e-8:
    foxf1_fixedwidth_peak = 1.0

foxf1_equal_support_peak = float(np.nanmax(foxf1_equal_support_across_images_df["mean"].to_numpy(dtype=float))) if len(foxf1_equal_support_across_images_df) else 1.0
if not np.isfinite(foxf1_equal_support_peak) or foxf1_equal_support_peak <= 1e-8:
    foxf1_equal_support_peak = 1.0

foxf1_pooled_fixedwidth_peak = float(np.nanmax(foxf1_merged_df["weighted_mean_value"].to_numpy(dtype=float))) if len(foxf1_merged_df) else 1.0
if not np.isfinite(foxf1_pooled_fixedwidth_peak) or foxf1_pooled_fixedwidth_peak <= 1e-8:
    foxf1_pooled_fixedwidth_peak = 1.0

foxf1_pooled_equal_support_peak = float(np.nanmax(foxf1_equal_support_df["mean_value"].to_numpy(dtype=float))) if len(foxf1_equal_support_df) else 1.0
if not np.isfinite(foxf1_pooled_equal_support_peak) or foxf1_pooled_equal_support_peak <= 1e-8:
    foxf1_pooled_equal_support_peak = 1.0

foxf1_plot_fixedwidth_per_df = foxf1_stats_df.copy().sort_values(["canonical_position", "bin_mid_um"]).reset_index(drop=True)
for col in ["mean_value", "median_value", "std_value", "sem_value"]:
    if col in foxf1_plot_fixedwidth_per_df.columns:
        foxf1_plot_fixedwidth_per_df[col] = foxf1_plot_fixedwidth_per_df[col].astype(float) / foxf1_fixedwidth_peak

foxf1_plot_main_agg = foxf1_trace_df.copy().sort_values("bin_mid_um").reset_index(drop=True)
for col in ["mean", "sd", "sem"]:
    if col in foxf1_plot_main_agg.columns:
        foxf1_plot_main_agg[col] = foxf1_plot_main_agg[col].astype(float) / foxf1_fixedwidth_peak
foxf1_plot_main_agg["ci95"] = 1.96 * foxf1_plot_main_agg["sem"].astype(float)

foxf1_plot_equal_support_per_df = foxf1_stats_df.copy().sort_values(["canonical_position", "bin_mid_um"]).reset_index(drop=True)
for col in ["mean_value", "median_value", "std_value", "sem_value"]:
    if col in foxf1_plot_equal_support_per_df.columns:
        foxf1_plot_equal_support_per_df[col] = foxf1_plot_equal_support_per_df[col].astype(float) / foxf1_equal_support_peak

foxf1_plot_alt_agg = foxf1_equal_support_across_images_df.copy().sort_values("bin_mid_um").reset_index(drop=True)
for col in ["mean", "sd", "sem"]:
    if col in foxf1_plot_alt_agg.columns:
        foxf1_plot_alt_agg[col] = foxf1_plot_alt_agg[col].astype(float) / foxf1_equal_support_peak
foxf1_plot_alt_agg["ci95"] = 1.96 * foxf1_plot_alt_agg["sem"].astype(float)
foxf1_equal_support_xmin = 50.0

eq_ci_window = foxf1_plot_alt_agg[foxf1_plot_alt_agg["bin_mid_um"] <= FOXF1_XMAX].copy()
if len(eq_ci_window) > 0:
    eq_ci_lo = float(np.nanmin(eq_ci_window["mean"] - eq_ci_window["ci95"]))
    eq_ci_hi = float(np.nanmax(eq_ci_window["mean"] + eq_ci_window["ci95"]))
    foxf1_equal_support_ylim = (
        float(np.floor((eq_ci_lo - 0.05) / 0.05) * 0.05),
        float(np.ceil((eq_ci_hi + 0.05) / 0.05) * 0.05),
    )
else:
    foxf1_equal_support_ylim = (-0.15, 1.35)

foxf1_fixedwidth_ylim = (-0.10, 1.35)

foxf1_plot_fixedwidth_merged = foxf1_merged_df.copy().sort_values("bin_mid_um").reset_index(drop=True)
for col in ["weighted_mean_value", "pooled_sd_value", "pooled_sem_value"]:
    if col in foxf1_plot_fixedwidth_merged.columns:
        foxf1_plot_fixedwidth_merged[col] = foxf1_plot_fixedwidth_merged[col].astype(float) / foxf1_pooled_fixedwidth_peak

foxf1_plot_equal_support_merged = foxf1_equal_support_df.copy().sort_values("bin_mid_um").reset_index(drop=True)
for col in ["mean_value", "median_value", "std_value", "sem_value"]:
    if col in foxf1_plot_equal_support_merged.columns:
        foxf1_plot_equal_support_merged[col] = foxf1_plot_equal_support_merged[col].astype(float) / foxf1_pooled_equal_support_peak
foxf1_equal_support_pooled_xmin = 50.0

fx_pool_window = foxf1_plot_fixedwidth_merged[foxf1_plot_fixedwidth_merged["bin_mid_um"] <= FOXF1_XMAX].copy()
if len(fx_pool_window) > 0:
    fx_lo = float(np.nanmin(fx_pool_window["weighted_mean_value"] - fx_pool_window["pooled_sd_value"]))
    fx_hi = float(np.nanmax(fx_pool_window["weighted_mean_value"] + fx_pool_window["pooled_sd_value"]))
    foxf1_pooled_fixedwidth_ylim = (
        float(np.floor((fx_lo - 0.05) / 0.05) * 0.05),
        float(np.ceil((fx_hi + 0.05) / 0.05) * 0.05),
    )
else:
    foxf1_pooled_fixedwidth_ylim = (-0.5, 1.8)

eq_pool_window = foxf1_plot_equal_support_merged[foxf1_plot_equal_support_merged["bin_mid_um"] <= FOXF1_XMAX].copy()
if len(eq_pool_window) > 0:
    eq_lo = float(np.nanmin(eq_pool_window["mean_value"] - eq_pool_window["std_value"]))
    eq_hi = float(np.nanmax(eq_pool_window["mean_value"] + eq_pool_window["std_value"]))
    foxf1_pooled_equal_support_ylim = (
        float(np.floor((eq_lo - 0.05) / 0.05) * 0.05),
        float(np.ceil((eq_hi + 0.05) / 0.05) * 0.05),
    )
else:
    foxf1_pooled_equal_support_ylim = (-0.8, 2.3)

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_fixed_foxf1_trace(
    ax=ax,
    per_image_df=foxf1_plot_equal_support_per_df,
    agg_df=foxf1_plot_alt_agg,
    x_col="bin_mid_um",
    y_col="mean",
    band_col="ci95",
    title="Fixed FOXF1-YFP reporter (fixed) / DAPI by nearest-bead distance | equal-support bins",
    x_label="Mean distance within equal-support bin (um)",
    aggregate_label="Across-image mean",
    band_label="Across-image mean 95% CI",
)
ax.set_ylim(*foxf1_equal_support_ylim)
ax.set_xlim(foxf1_equal_support_xmin, FOXF1_XMAX)
_label_panel(ax, "9.1")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_TRACE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_fixed_foxf1_trace(
    ax=ax,
    per_image_df=foxf1_plot_fixedwidth_per_df,
    agg_df=foxf1_plot_main_agg,
    x_col="bin_mid_um",
    y_col="mean",
    band_col="ci95",
    title="Fixed FOXF1-YFP reporter (fixed) / DAPI by nearest-bead distance | fixed-width bins",
    aggregate_label="Across-image mean",
    band_label="Across-image mean 95% CI",
)
ax.set_ylim(*foxf1_fixedwidth_ylim)
ax.set_xlim(0.0, FOXF1_XMAX)
_label_panel(ax, "9.2")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_TRACE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

foxf1_delta_df = foxf1_plot_fixedwidth_per_df[["canonical_position", "bin_mid_um", "mean_value"]].merge(
    foxf1_plot_main_agg[["bin_mid_um", "mean"]],
    on="bin_mid_um",
    how="inner",
)
foxf1_delta_df["delta"] = foxf1_delta_df["mean_value"] - foxf1_plot_main_agg.set_index("bin_mid_um").reindex(foxf1_delta_df["bin_mid_um"])["mean"].to_numpy(dtype=float)
foxf1_dip_window_df = foxf1_delta_df[
    foxf1_delta_df["bin_mid_um"].between(FOXF1_DIP_WINDOW_UM[0], FOXF1_DIP_WINDOW_UM[1], inclusive="both")
].copy()
foxf1_dip_drivers = (
    foxf1_dip_window_df.groupby("canonical_position", as_index=False)["delta"]
    .mean()
    .sort_values("delta", ascending=True)["canonical_position"]
    .astype(str)
    .tolist()[:2]
)
foxf1_primary_dip_driver = foxf1_dip_drivers[0]
foxf1_review_specs = [
    {"pos": FOXF1_REVIEW_POSITION, "color": "#1f77b4", "label_prefix": "Requested review trace"},
    {"pos": foxf1_dip_drivers[0], "color": "#d62728", "label_prefix": f"Primary dip near {int(round(np.mean(FOXF1_DIP_WINDOW_UM)))} um"},
    {"pos": foxf1_dip_drivers[1], "color": "#ff7f0e", "label_prefix": f"Secondary dip near {int(round(np.mean(FOXF1_DIP_WINDOW_UM)))} um"},
]
foxf1_review_specs_unique = []
seen_positions = set()
for spec in foxf1_review_specs:
    pos = str(spec["pos"])
    if pos in seen_positions:
        continue
    spec = dict(spec)
    spec["pos"] = pos
    foxf1_review_specs_unique.append(spec)
    seen_positions.add(pos)

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_fixed_foxf1_trace(
    ax=ax,
    per_image_df=foxf1_plot_fixedwidth_per_df,
    agg_df=foxf1_plot_main_agg,
    x_col="bin_mid_um",
    y_col="mean",
    band_col="ci95",
    title="Fixed FOXF1-YFP reporter (fixed) / DAPI by nearest-bead distance | fixed-width bins (selected trace review)",
    aggregate_label="Across-image mean",
    band_label="Across-image mean 95% CI",
)
for spec in foxf1_review_specs_unique:
    pos = spec["pos"]
    sub = foxf1_plot_fixedwidth_per_df[foxf1_plot_fixedwidth_per_df["canonical_position"] == pos].copy().sort_values("bin_mid_um")
    if len(sub) == 0:
        continue
    ax.plot(
        sub["bin_mid_um"],
        sub["mean_value"],
        color=spec["color"],
        linewidth=2.4,
        alpha=0.98,
        zorder=5,
        label=f"{spec['label_prefix']}: {pos}",
    )
    ax.scatter(
        sub["bin_mid_um"],
        sub["mean_value"],
        color=spec["color"],
        edgecolor="white",
        linewidth=0.4,
        s=20,
        alpha=0.98,
        zorder=6,
    )
    finite = sub[np.isfinite(sub["mean_value"].to_numpy(dtype=float))]
    if len(finite) > 0:
        visible = finite[finite["bin_mid_um"] <= FOXF1_REVIEW_XMAX + 1e-9]
        visible_for_label = visible if len(visible) > 0 else finite
        if pos == foxf1_primary_dip_driver:
            focus = visible_for_label[visible_for_label["bin_mid_um"] >= max(0.0, FOXF1_REVIEW_XMAX - 80.0)]
            finite_for_label = focus if len(focus) > 0 else visible_for_label
            idx_lab = int(np.argmin(finite_for_label["mean_value"].to_numpy(dtype=float)))
        else:
            idx_lab = int(np.argmax(np.abs(visible_for_label["mean_value"].to_numpy(dtype=float) - np.interp(visible_for_label["bin_mid_um"].to_numpy(dtype=float), foxf1_plot_main_agg["bin_mid_um"].to_numpy(dtype=float), foxf1_plot_main_agg["mean"].to_numpy(dtype=float)))))
            finite_for_label = visible_for_label
        x_lab = float(finite_for_label["bin_mid_um"].iloc[idx_lab])
        y_lab = float(finite_for_label["mean_value"].iloc[idx_lab])
        if x_lab >= FOXF1_REVIEW_XMAX - 45.0:
            x_text = x_lab - 10.0
            ha = "right"
        else:
            x_text = x_lab + 10.0
            ha = "left"
        ax.text(
            x_text,
            y_lab,
            pos,
            color=spec["color"],
            fontsize=9,
            fontweight="bold",
            va="center",
            ha=ha,
            bbox={"facecolor": "white", "edgecolor": spec["color"], "boxstyle": "round,pad=0.18", "alpha": 0.85},
            zorder=7,
            clip_on=True,
        )
ax.set_ylim(-0.10, 1.30)
ax.set_xlim(0.0, FOXF1_XMAX)
_label_panel(ax, "9.3")
ax.legend(loc="best", frameon=False)
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_TRACE_FIXED_WIDTH_HIGHLIGHTED_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 4.8))
bar_widths = (foxf1_merged_df["bin_end_um"] - foxf1_merged_df["bin_start_um"]).astype(float).to_numpy()
ax.bar(foxf1_merged_df["bin_mid_um"], foxf1_merged_df["total_count_px"], width=bar_widths, color="0.7", edgecolor="0.45", linewidth=0.5, align="center")
ax.set_title("DAPI-gated analysis support | total retained pixels pooled across images, fixed-width distance bins")
ax.set_xlabel("Distance from nearest bead (um)")
ax.set_ylabel("Total retained pixels in bin")
_label_panel(ax, "9.4")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_ANALYSIS_SUPPORT_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
eq_mean = foxf1_plot_equal_support_merged["mean_value"].to_numpy(dtype=float)
eq_sd = foxf1_plot_equal_support_merged["std_value"].to_numpy(dtype=float)
ax.plot(foxf1_plot_equal_support_merged["bin_mid_um"], eq_mean, color=MEASUREMENT_COLORS[FOXF1_FIXED_MEAS_NAME], lw=2.2, marker="o", ms=3, zorder=3, label="Mean trace")
ax.fill_between(foxf1_plot_equal_support_merged["bin_mid_um"], eq_mean - eq_sd, eq_mean + eq_sd, color=MEASUREMENT_COLORS[FOXF1_FIXED_MEAS_NAME], alpha=0.18, zorder=2, label="±SD")
ax.set_title("Fixed FOXF1-YFP reporter (fixed) / DAPI vs distance | all retained pixels merged, equal-support bins (peak-normalized)")
ax.set_xlabel("Mean distance within equal-support bin (um)")
ax.set_ylabel("Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)")
ax.legend(loc="upper left")
ax.text(0.98, 0.95, f"Target pixels/bin = {equal_support_target_px:,}\nBins = {len(foxf1_plot_equal_support_merged):,}", transform=ax.transAxes, ha="right", va="top", fontsize=8, bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9})
ax.set_xlim(foxf1_equal_support_pooled_xmin, FOXF1_XMAX)
ax.set_ylim(*foxf1_pooled_equal_support_ylim)
_label_panel(ax, "9.5")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_MERGED_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
merged_mean = foxf1_plot_fixedwidth_merged["weighted_mean_value"].to_numpy(dtype=float)
merged_sd = foxf1_plot_fixedwidth_merged["pooled_sd_value"].to_numpy(dtype=float)
ax.plot(foxf1_plot_fixedwidth_merged["bin_mid_um"], merged_mean, color=MEASUREMENT_COLORS[FOXF1_FIXED_MEAS_NAME], lw=2.2, marker="o", ms=3, zorder=3, label="Mean trace")
ax.fill_between(foxf1_plot_fixedwidth_merged["bin_mid_um"], merged_mean - merged_sd, merged_mean + merged_sd, color=MEASUREMENT_COLORS[FOXF1_FIXED_MEAS_NAME], alpha=0.18, zorder=2, label="±SD")
ax.set_title("Fixed FOXF1-YFP reporter (fixed) / DAPI vs distance | all retained pixels merged, fixed-width bins (peak-normalized)")
ax.set_xlabel("Distance from nearest bead (um)")
ax.set_ylabel("Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)")
ax.legend(loc="upper left")
ax.text(0.98, 0.95, f"Median pixels/bin = {equal_support_target_px:,}", transform=ax.transAxes, ha="right", va="top", fontsize=8, bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9})
ax.set_xlim(0.0, FOXF1_XMAX)
ax.set_ylim(*foxf1_pooled_fixedwidth_ylim)
_label_panel(ax, "9.6")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_MERGED_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()



### Live Vs Fixed FOXF1

Direct overlay of live and fixed FOXF1 on a common x-axis with separate y-axes, showing equal-support first.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_live_vs_fixed_overlay(ax=ax, live_df=live_equal_support_plot_df, fixed_df=fixed_equal_support_plot_df, title="FOXF1-YFP reporter activity | live vs fixed | equal-support bins", x_label="Mean distance within equal-support bin (um)")
_label_panel(ax, "9.7")
plt.tight_layout()
_save_plot_all_formats(fig, FOXF1_LIVE_VS_FIXED_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
_plot_live_vs_fixed_overlay(ax=ax, live_df=live_fixedwidth_plot_df, fixed_df=fixed_fixedwidth_plot_df, title="FOXF1-YFP reporter activity | live vs fixed | fixed-width bins", x_label="Distance from nearest bead (um)")
_label_panel(ax, "9.8")
plt.tight_layout()
_save_plot_all_formats(fig, FOXF1_LIVE_VS_FIXED_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()



## Fixed Multi-Channel Bead-Distance Views

Once the FOXF1 choice is in view, these plots show how the fixed channels behave together and separately in bead-distance coordinates.

### Three-Channel Overview

FOXF1-YFP reporter (fixed), SOX2, and TBXT overlaid, shown as individual final images only, with equal-support first.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_trace(
    ax=ax,
    per_image_df=plot_fixedwidth_per_df_window,
    agg_df=plot_alt_agg_window,
    title="Fixed FOXF1-YFP reporter (fixed), SOX2, and TBXT by nearest-bead distance | equal-support bins",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
ax.set_ylim(*foxf1_equal_support_ylim)
ax.set_xlim(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX)
_label_panel(ax, "9.9")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_RATIO_TRACE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_trace(
    ax=ax,
    per_image_df=plot_fixedwidth_per_df_window,
    agg_df=plot_main_agg_window,
    title="Fixed FOXF1-YFP reporter (fixed), SOX2, and TBXT by nearest-bead distance | fixed-width bins",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
)
ax.set_ylim(*foxf1_fixedwidth_ylim)
ax.set_xlim(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX)
_label_panel(ax, "9.10")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_RATIO_TRACE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_merged(
    ax=ax,
    merged_df=plot_equal_support_merged,
    title="Fixed ratios vs distance | all retained pixels merged, equal-support bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
    mean_col="mean_value",
    sd_col="std_value",
    band_mode="sd",
)
ax.set_ylim(-0.2, 1.2)
ax.set_xlim(50.0, MULTICHANNEL_TRACE_XMAX)
ax.text(0.98, 0.95, f"Target pixels/bin = {equal_support_target_px:,}\nBins = {len(plot_equal_support_merged) // len(MEASUREMENT_ORDER):,}", transform=ax.transAxes, ha="right", va="top", fontsize=8, bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9})
_label_panel(ax, "9.11")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_RATIO_MERGED_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_multi_channel_merged(
    ax=ax,
    merged_df=plot_fixedwidth_merged,
    title="Fixed ratios vs distance | all retained pixels merged, fixed-width bins (peak-normalized)",
    ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)",
    mean_col="weighted_mean_value",
    sd_col="pooled_sd_value",
    band_mode="sd",
)
ax.set_ylim(-0.2, 1.2)
ax.set_xlim(0.0, MULTICHANNEL_TRACE_XMAX)
ax.text(0.98, 0.95, f"Median pixels/bin = {equal_support_target_px:,}", transform=ax.transAxes, ha="right", va="top", fontsize=8, bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25", "alpha": 0.9})
_label_panel(ax, "9.12")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_RATIO_MERGED_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()



### FOXF1 And SOX2 Overlay

Reduced two-channel version of the main fixed plot when TBXT is distracting or biologically harder to interpret in bead-distance space, again with equal-support first.

In [ ]:
FOXF1_SOX2_MEASUREMENT_ORDER = ["fixed_tagyfp_bgz_over_dapi_gate", "fixed_sox2_bgz_over_dapi_gate"]
foxf1_sox2_plot_fixedwidth_per_df = plot_fixedwidth_per_df_window[plot_fixedwidth_per_df_window["measurement_name"].isin(FOXF1_SOX2_MEASUREMENT_ORDER)].copy()
foxf1_sox2_plot_main_agg = plot_main_agg_window[plot_main_agg_window["measurement_name"].isin(FOXF1_SOX2_MEASUREMENT_ORDER)].copy()
foxf1_sox2_plot_alt_agg = plot_alt_agg_window[plot_alt_agg_window["measurement_name"].isin(FOXF1_SOX2_MEASUREMENT_ORDER)].copy()

fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_subset_channel_trace(ax=ax, per_image_df=foxf1_sox2_plot_fixedwidth_per_df, agg_df=foxf1_sox2_plot_alt_agg, measurement_order=FOXF1_SOX2_MEASUREMENT_ORDER, title="Fixed FOXF1-YFP reporter (fixed) and SOX2 by nearest-bead distance | equal-support bins (peak-normalized)", ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)")
ax.set_ylim(*foxf1_equal_support_ylim)
ax.set_xlim(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX)
_label_panel(ax, "9.13")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_SOX2_TRACE_EQUAL_SUPPORT_PNG, dpi=180, bbox_inches="tight")
plt.show()
fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
_plot_subset_channel_trace(ax=ax, per_image_df=foxf1_sox2_plot_fixedwidth_per_df, agg_df=foxf1_sox2_plot_main_agg, measurement_order=FOXF1_SOX2_MEASUREMENT_ORDER, title="Fixed FOXF1-YFP reporter (fixed) and SOX2 by nearest-bead distance | fixed-width bins (peak-normalized)", ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)")
ax.set_ylim(*foxf1_fixedwidth_ylim)
ax.set_xlim(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX)
_label_panel(ax, "9.14")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_SOX2_TRACE_FIXED_WIDTH_PNG, dpi=180, bbox_inches="tight")
plt.show()



### SOX2 And TBXT Single-Channel Breakouts

Channel-isolated views for the fixed markers that are not already covered by the FOXF1-only section, with equal-support first.

In [ ]:
SINGLE_CHANNEL_OUTPUTS = {
    "fixed_sox2_bgz_over_dapi_gate": {"fixed_width_png": FIXED_SOX2_TRACE_FIXED_WIDTH_PNG, "equal_support_png": FIXED_SOX2_TRACE_EQUAL_SUPPORT_PNG, "labels": ("9.15", "9.16")},
    "fixed_t_bgz_over_dapi_gate": {"fixed_width_png": FIXED_T_TRACE_FIXED_WIDTH_PNG, "equal_support_png": FIXED_T_TRACE_EQUAL_SUPPORT_PNG, "labels": ("9.17", "9.18")},
}

for meas_name in ["fixed_sox2_bgz_over_dapi_gate", "fixed_t_bgz_over_dapi_gate"]:
    color = MEASUREMENT_COLORS[meas_name]
    label = MEASUREMENT_LABELS[meas_name]
    out = SINGLE_CHANNEL_OUTPUTS[meas_name]

    fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
    _plot_single_fixed_channel_trace(ax=ax, per_image_df=plot_fixedwidth_per_df_window, agg_df=plot_alt_agg_window, meas_name=meas_name, title=f"{label} by nearest-bead distance | equal-support bins (peak-normalized)", ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)", color=color)
    channel_ylim = single_channel_ylim_window_by_meas.get(meas_name)
    if channel_ylim is not None:
        ax.set_ylim(*channel_ylim)
    ax.set_xlim(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX)
    _label_panel(ax, out["labels"][0])
    plt.tight_layout()
    _save_plot_all_formats(fig, out["equal_support_png"], dpi=180, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(1, 1, figsize=(7.4, 5.8))
    _plot_single_fixed_channel_trace(ax=ax, per_image_df=plot_fixedwidth_per_df_window, agg_df=plot_main_agg_window, meas_name=meas_name, title=f"{label} by nearest-bead distance | fixed-width bins (peak-normalized)", ylabel="Normalized mean standardized-channel / raw DAPI\n(channel peak bin = 1)", color=color)
    channel_ylim = single_channel_ylim_window_by_meas.get(meas_name)
    if channel_ylim is not None:
        ax.set_ylim(*channel_ylim)
    ax.set_xlim(MULTICHANNEL_TRACE_XMIN, MULTICHANNEL_TRACE_XMAX)
    _label_panel(ax, out["labels"][1])
    plt.tight_layout()
    _save_plot_all_formats(fig, out["fixed_width_png"], dpi=180, bbox_inches="tight")
    plt.show()


## Fixed Pairwise State-Space Plots

Pooled analysis pixels visualized as density plots to show how channels co-vary independent of distance binning.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 5.6))
_plot_pairwise_density(ax=ax, x=sox2_vals_for_foxf1_pair, y=foxf1_vals, xlabel="SOX2 / DAPI", ylabel="FOXF1-YFP reporter (fixed) / DAPI", title="Fixed analysis pixels | SOX2 vs FOXF1-YFP reporter")
_label_panel(ax, "9.19")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_FOXF1_SOX2_DENSITY_PNG, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(6.4, 5.6))
_plot_pairwise_density(ax=ax, x=sox2_vals_for_t_pair, y=t_vals_for_sox2_pair, xlabel="SOX2 / DAPI", ylabel="TBXT / DAPI", title="Fixed analysis pixels | SOX2 vs TBXT")
_label_panel(ax, "9.20")
plt.tight_layout()
_save_plot_all_formats(fig, FIXED_SOX2_T_DENSITY_PNG, dpi=180, bbox_inches="tight")
plt.show()
